In [1]:
%load_ext kedro.ipython

The kedro.ipython extension is already loaded. To reload it, use:
  %reload_ext kedro.ipython


In [2]:
!uv pip install --upgrade seaborn matplotlib

Using Python 3.12.7 environment at: C:\Users\arczi\source\repos\Mini\Msi\wsi-reg-cls\wsi-reg\.venv
Resolved 14 packages in 159ms
Prepared 5 packages in 2ms
Uninstalled 5 packages in 878ms
Installed 5 packages in 1.11s
 - fonttools==4.61.1
 + fonttools==4.62.1
 - kiwisolver==1.4.9
 + kiwisolver==1.5.0
 - numpy==2.4.2
 + numpy==2.4.3
 - pandas==2.3.3
 + pandas==3.0.1
 - seaborn==0.12.2
 + seaborn==0.13.2


In [3]:
import pandas as pd

df = catalog.load("domy")

df = df.sort_index(axis=1)

[03/22/26 19:53:07] INFO     Loading data from domy (CSVDataset)...                            ]8;id=854090;file://C:\Users\arczi\source\repos\Mini\Msi\wsi-reg-cls\wsi-reg\.venv\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=1956;file://C:\Users\arczi\source\repos\Mini\Msi\wsi-reg-cls\wsi-reg\.venv\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [4]:
df['Alley'] = df['Alley'].replace('?', 'NoAlley')

In [5]:
import numpy as np

kolumny_piwnica = ['BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'BsmtQual']

for kol in kolumny_piwnica:
    df[kol] = df[kol].replace('?', np.nan)

maska_brak_piwnicy = df['BsmtQual'].isna()

for kol in kolumny_piwnica:
    df.loc[maska_brak_piwnicy, kol] = 0

for kol in ['BsmtExposure', 'BsmtFinType2']:
    najczestsza = df[kol].mode()[0]
    df[kol] = df[kol].fillna(najczestsza)

In [6]:
kol = 'Electrical'
zmiana = df[kol].mode()[0]
df[kol] = df[kol].replace('?', zmiana)

In [7]:
kol = 'Fence'
zmiana = 'NoFence'
df[kol] = df[kol].replace('?', zmiana)

In [8]:
df.loc[df['Fireplaces'] == 0, 'FireplaceQu'] = 'None'

In [9]:
import pandas as pd

garage_categorical = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']

for col in garage_categorical:
    df[col] = df[col].replace('?', 'None')

df['GarageYrBlt'] = df['GarageYrBlt'].replace('?', 0)
df['GarageYrBlt'] = pd.to_numeric(df['GarageYrBlt'])

In [10]:
import numpy as np

df['LotFrontage'] = df['LotFrontage'].replace('?', np.nan)
df['LotFrontage'] = pd.to_numeric(df['LotFrontage'])

df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"].transform(
    lambda x: x.fillna(x.median())
)

In [11]:
most_frequent_type = df['MasVnrType'].mode()[0]

df.loc[(df['MasVnrType'].isnull()) & ((df['MasVnrArea'] == 0) | (df['MasVnrArea'] == 1)), 'MasVnrType'] = 'None'
df['MasVnrType'] = df['MasVnrType'].replace('?', 'None')
df['MasVnrArea'] = df['MasVnrArea'].replace('?', 0)
df['MasVnrArea'] = pd.to_numeric(df['MasVnrArea'])
df['MasVnrType'] = df['MasVnrType'].fillna(most_frequent_type)

In [12]:
kol = 'MiscFeature'
zmiana = 'None'
df[kol] = df[kol].replace('?', zmiana)

In [13]:
kol = 'PoolQC'
zmiana = 'None'
df[kol] = df[kol].replace('?', zmiana)

In [14]:
zliczone_pytajniki = (df == '?').sum()

# 2. Wyświetlamy tylko te kolumny, w których wynik jest większy od zera
tylko_z_pytajnikiem = zliczone_pytajniki[zliczone_pytajniki > 0]

print(tylko_z_pytajnikiem)

Series([], dtype: int64)


In [15]:
braki = df.isna().sum()

# Wyświetlamy tylko te kolumny, gdzie liczba braków jest > 0
print(braki[braki > 0])

Series([], dtype: int64)


In [16]:
with pd.option_context('display.max_rows', None):
    print(df.dtypes)

1stFlrSF           int64
2ndFlrSF           int64
3SsnPorch          int64
Alley             object
BedroomAbvGr       int64
BldgType          object
BsmtCond          object
BsmtExposure      object
BsmtFinSF1         int64
BsmtFinSF2         int64
BsmtFinType1      object
BsmtFinType2      object
BsmtFullBath       int64
BsmtHalfBath       int64
BsmtQual          object
BsmtUnfSF          int64
CentralAir        object
Condition1        object
Condition2        object
Electrical        object
EnclosedPorch      int64
ExterCond         object
ExterQual         object
Exterior1st       object
Exterior2nd       object
Fence             object
FireplaceQu       object
Fireplaces         int64
Foundation        object
FullBath           int64
Functional        object
GarageArea         int64
GarageCars         int64
GarageCond        object
GarageFinish      object
GarageQual        object
GarageType        object
GarageYrBlt      float64
GrLivArea          int64
HalfBath           int64


In [23]:
df['ScreenPorch'].value_counts(normalize=True)


ScreenPorch
0      0.920548
192    0.004110
120    0.003425
224    0.003425
189    0.002740
         ...   
155    0.000685
220    0.000685
119    0.000685
165    0.000685
40     0.000685
Name: proportion, Length: 76, dtype: float64